In [1]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import plotly.graph_objects as go # 그래프 생성의 디테일 제어 

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [57]:
waste_df = pd.read_csv("../data/전국폐기물배출업체표준데이터.csv",encoding="cp949")
# waste_df.head()
# waste_df.tail()
# waste_df.info
waste_df.dtypes
# waste_df.shape
# waste_df.describe

사업장명            str
소재지도로명주소        str
소재지지번주소         str
위도          float64
경도          float64
폐기물분류명          str
폐기물세부종류명        str
처리방법내용          str
전화번호            str
배출량         float64
신고기준연도        int64
관리기관명           str
관리기관전화번호        str
데이터기준일자         str
제공기관코드        int64
제공기관명           str
dtype: object

In [ ]:
analysis_df = waste_df[ # 분석 데이터 프레임의 컬럼을 지정하여 정제하는 작업
    [
        "사업장명",
        "제공기관명",
        "폐기물분류명",
        "배출량"
    ]
].copy()

,사업장명,제공기관명,폐기물분류명,배출량
0,경상남도소방본부119특수대응단119항공대,경상남도 합천군,의료폐기물,0.360
1,합천축협 유전자원센터,경상남도 합천군,의료폐기물,1.500
2,대병구급대,경상남도 합천군,의료폐기물,0.360
3,덕곡지역대,경상남도 합천군,의료폐기물,0.360
4,북부119안전센터,경상남도 합천군,의료폐기물,0.360
...,...,...,...,...
4237,한국지역난방공사,대구광역시 달서구,사업장폐기물,702091.666
4238,(주)금복주,대구광역시 달서구,사업장폐기물,501000.000
4239,삼화식품공사,대구광역시 달서구,사업장폐기물,135000.000
4240,계명대학교,대구광역시 달서구,사업장폐기물,62000.000


In [12]:
analysis_df["배출량"] = pd.to_numeric( #?
    analysis_df["배출량"],
    errors="coerce"
)

analysis_df["사업장명"] = analysis_df["사업장명"].str.strip() # 문자 공백제거
analysis_df["제공기관명"] = analysis_df["제공기관명"].str.strip() # 문자 공백제거

print(analysis_df.isna().sum()) # 결측치 값 합
print("중복 행:", analysis_df.duplicated().sum())

사업장명      0
제공기관명     0
폐기물분류명    0
배출량       0
dtype: int64
중복 행: 118


In [ ]:
clean_df = analysis_df.dropna(
    subset=["제공기관명", "배출량"] # 컬럼 지정 및 결측치 제거
).copy()

In [ ]:
summary_df = ( #컬럼 지정 및 통계
    clean_df
    .groupby("제공기관명", as_index=False)
    .agg(
        총배출량=("배출량", "sum"),
        평균배출량=("배출량", "mean"),
        중앙배출량=("배출량", "median"),
        사업장수=("사업장명", "nunique")
    )
    .sort_values("총배출량", ascending=False)
)

summary_df.head(10)

,제공기관명,총배출량,평균배출량,중앙배출량,사업장수
5,대구광역시 달서구,7.374182e+07,59373.444172,277.5000,1185
3,경기도 수원시,1.748529e+07,90597.352332,14201.0000,193
1,경기도 군포시,1.579872e+07,125386.666523,12250.0000,125
6,대구광역시 중구,7.376535e+06,15529.547368,207.0000,469
9,전북특별자치도 진안군,3.623532e+06,11880.432787,30.0000,101
2,경기도 부천시,1.546462e+06,2294.454175,200.0000,154
10,제주특별자치도 서귀포시,2.953944e+05,1893.553615,284.0000,148
0,강원특별자치도 정선군,1.096697e+05,294.811011,3.6000,138
4,경상남도 합천군,1.029182e+05,631.399936,0.5316,154
7,부산광역시 금정구,7.675175e+04,227.076183,20.0000,332


In [16]:
print("원본 행 수:", waste_df.shape[0])
print("분석 행 수:", analysis_df.shape[0])
print("정제 후 행 수:", clean_df.shape[0])

display(summary_df.head())
display(summary_df.isna().sum())

원본 행 수: 4242
분석 행 수: 4242
정제 후 행 수: 4242


,제공기관명,총배출량,평균배출량,중앙배출량,사업장수
5,대구광역시 달서구,7.374182e+07,59373.444172,277.5,1185
3,경기도 수원시,1.748529e+07,90597.352332,14201.0,193
1,경기도 군포시,1.579872e+07,125386.666523,12250.0,125
6,대구광역시 중구,7.376535e+06,15529.547368,207.0,469
9,전북특별자치도 진안군,3.623532e+06,11880.432787,30.0,101


제공기관명    0
총배출량     0
평균배출량    0
중앙배출량    0
사업장수     0
dtype: int64

In [17]:
import plotly.express as px

top10_df = summary_df.head(10)

fig = px.bar(
    top10_df,
    x="제공기관명",
    y="총배출량",
    text_auto=".1f",
    title="제공기관별 총 폐기물 배출량 상위 10곳",
    template="plotly_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    title_x=0.5,
    xaxis_title="제공기관명",
    yaxis_title="총배출량"
)

fig.show()

In [23]:
# fig = px.scatter(
#     data_frame=summary_df,
#     x="제공기관명",
#     y="총배출량",
#     trendline="ols",
#     trendline_color_override="#E45756",
#     title="제공기관과 총 배출량 상관관계",
#     labels={
#         "제공기관명": "제공기관명 (시간)",
#         "총배출량": "총배출량"
#     },
#     template="plotly_white",
#     hover_data={
#         "제공기관명": ":.1f",
#         "총배출량": ":.1f"
#     }
# )

fig = px.bar(
    data_frame=summary_df.head(10),
    x="제공기관명",
    y="총배출량",
    text_auto=".1f",
    title="제공기관별 총배출량 상위 10곳",
    template="plotly_white"
)
fig.show()


fig.update_traces(
    textposition="outside",
    marker_color="#4C78A8"
)

fig.update_layout(
    width=900,
    height=500,
    title_x=0.5
)

fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(
        size=11,
        color="#4C78A8",
        opacity=0.8,
        line=dict(
            width=1,
            color="white"
        )
    )
)

fig.update_layout(
    width=850,
    height=550,
    title=dict(
        text="제공기관과 총배출량 관계",
        x=0.5,
        xanchor="center",
        font=dict(size=22)
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#333333"
    ),
    plot_bgcolor="#F8FAFC",
    paper_bgcolor="white",
    xaxis=dict(
        showgrid=True,
        gridcolor="#D9E2EC",
        zeroline=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#0D7D8100",
        zeroline=False
    ),
    legend=dict(
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig.show()


In [ ]:
daegu_df = clean_df[
    clean_df["제공기관명"].str.contains("대구", na=False) # 현재 대구광역시가 사업장이 많아 이상치로 잡혀 추가적인 파악이 필요
].copy()

In [ ]:
#대구광역시 컬럼 기준으로 높은 것 찾기
## 1. 제공기관별 총배출량 Top 10
region_total_df = (
    clean_df 
    .groupby("제공기관명", as_index=False)
    .agg(총배출량=("배출량", "sum"))
    .sort_values("총배출량", ascending=False)
    .head(10)
    .sort_values("총배출량")
)

fig = px.bar(
    region_total_df,
    x="총배출량",
    y="제공기관명",
    orientation="h",
    text_auto=".1f",
    title="제공기관별 총배출량 Top 10",
    labels={
        "제공기관명": "제공기관",
        "총배출량": "총배출량"
    },
    template="plotly_white"
)

fig.update_traces(
    textposition="outside",
    marker_color="#4C78A8"
)

fig.update_layout(
    width=900,
    height=500,
    title_x=0.5
)

fig.show()

In [ ]:
## 2. 제공기관별 평균배출량 Top 10

region_mean_df = (
    clean_df
    .groupby("제공기관명", as_index=False)
    .agg(평균배출량=("배출량", "mean"))
    .sort_values("평균배출량", ascending=False)
    .head(10)
    .sort_values("평균배출량")
)

fig = px.bar(
    region_mean_df,
    x="평균배출량",
    y="제공기관명",
    orientation="h",
    text_auto=".1f",
    title="제공기관별 평균배출량 Top 10",
    labels={
        "제공기관명": "제공기관",
        "평균배출량": "평균배출량"
    },
    template="plotly_white"
)

fig.update_traces(
    textposition="outside",
    marker_color="#F58518"
)

fig.update_layout(
    width=900,
    height=500,
    title_x=0.5
)

fig.show() # 제공기관별 평균배출량은 군포-수원-대구(달서)-대구(중구).....

In [42]:
## 3. 대구광역시 사업장별 배출량 Boxplot

# 대구광역시 데이터만 추출
daegu_df = clean_df[

#대구 데이터 추출, 0이하 값 제거, boxplot 분포, 상위 10개 사업장 막대그래프
clean_df["제공기관명"].str.contains("대구", na=False).copy()]


# 사업장별 배출량 합계
daegu_business_df = (
    daegu_df
    .groupby("사업장명", as_index=False) # group 사업장별 그룹화, 인덱스 번호 지정 X
    .agg(
        사업장별배출량=("배출량", "sum")# 배출량 합산 후 새 컬럼을 사업장별배출량 변수에 저장
    )
)

# 로그 스케일 사용을 위해 0 이하 값 제거
daegu_business_df = daegu_business_df[
    daegu_business_df["사업장별배출량"] > 0
].copy()


# -----------------------------
# 1. Boxplot
# -----------------------------
fig = px.box(
    daegu_business_df,
    y="사업장별배출량",
    points="all",
    title="대구광역시 사업장별 배출량 분포",
    labels={
        "사업장별배출량": "사업장별 배출량"
    },
    hover_data=["사업장명"],
    template="plotly_white"
)

fig.update_yaxes(
    type="log",
    title="사업장별 배출량 (로그 스케일)"
)

fig.update_traces(
    marker_color="#E45756",
    marker_size=6,
    line_color="#9E2A2B"
)

fig.update_layout(
    width=700,
    height=500,
    title_x=0.5
)

fig.show()


# -----------------------------
# 2. 배출량 상위 10개 사업장
# -----------------------------
top10 = (
    daegu_business_df
    .nlargest(10, "사업장별배출량")
    .sort_values("사업장별배출량")
)

fig = px.bar(
    top10,
    x="사업장별배출량",
    y="사업장명",
    orientation="h",
    title="대구광역시 배출량 상위 10개 사업장",
    labels={
        "사업장별배출량": "배출량",
        "사업장명": "사업장"
    },
    template="plotly_white",
    text="사업장별배출량"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    width=800,
    height=500,
    title_x=0.5
)

fig.show()

In [32]:
q1 = daegu_business_df["사업장별배출량"].quantile(0.25)
q3 = daegu_business_df["사업장별배출량"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Median:", daegu_business_df["사업장별배출량"].median())
print("Q3:", q3)
print("IQR:", iqr)
print("Upper Bound:", upper_bound)

Q1: 48.0
Median: 226.0
Q3: 7370.0
IQR: 7322.0
Upper Bound: 18353.0


In [ ]:
#이상치 확인
outliers = daegu_business_df[
    daegu_business_df["사업장별배출량"] > upper_bound
].sort_values(
    "사업장별배출량",
    ascending=False
)

outliers

,사업장명,사업장별배출량
475,대구공공시설관리공단 서부사업소,4.566141e+07
476,대구공공시설관리공단 성서사업소,2.246295e+06
106,(주)에스에이치,2.000000e+06
509,대구성서산업단지관리공단 환경사업소,1.888433e+06
86,(주)수성,1.555500e+06
...,...,...
985,송현효요양병원,1.918000e+04
257,계명문화대학교,1.860483e+04
502,대구서문교회,1.853000e+04
240,경북대학교치과병원,1.843600e+04


In [37]:
daegu_business_df.nlargest(20, "사업장별배출량")

,사업장명,사업장별배출량
475,대구공공시설관리공단 서부사업소,4.566141e+07
476,대구공공시설관리공단 성서사업소,2.246295e+06
106,(주)에스에이치,2.000000e+06
509,대구성서산업단지관리공단 환경사업소,1.888433e+06
86,(주)수성,1.555500e+06
503,대구서부하수슬러지건조연료화시설,1.540000e+06
1017,신대일페이퍼 주식회사,1.208333e+06
465,대구 서문시장 연합회,9.429300e+05
236,경북대학교병원,9.063140e+05
1604,현대백화점 대구점,9.012430e+05


In [38]:
max_business = daegu_business_df.loc[
    daegu_business_df["사업장별배출량"].idxmax(),
    "사업장명"
]

daegu_df[
    daegu_df["사업장명"] == max_business
]

,사업장명,제공기관명,폐기물분류명,배출량
3814,대구공공시설관리공단 서부사업소,대구광역시 달서구,지정폐기물,2.712508e+03
4236,대구공공시설관리공단 서부사업소,대구광역시 달서구,사업장폐기물,4.565870e+07


In [41]:
q1 = daegu_business_df["사업장별배출량"].quantile(0.25)
q3 = daegu_business_df["사업장별배출량"].quantile(0.75)

iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

normal_df = daegu_business_df[
    daegu_business_df["사업장별배출량"] <= upper_bound
]

normal_df

,사업장명,사업장별배출량
0,(복)대구가톨릭사회복지회부설 성심복지의원,22.000000
1,(사)대한산업보건협회 대구지역본부,202.500000
2,(의)구의료재단 구병원,13800.000000
3,(의)열경의료재단,549.000000
4,(의)열경의료재단 허병원,14000.000000
...,...,...
1642,후한의원,122.000000
1643,휴먼플러스,66.666667
1644,휴먼플러스(주),1441.666660
1647,흥일알티,10000.000000


In [ ]:
business_df = (
clean_df
.dropna(subset=["제공기관명", "사업장명", "배출량"])
.groupby(
    ["제공기관명", "사업장명"],
    as_index=False
)
.agg(
    사업장별배출량=("배출량", "sum")
)
)

business_df #사업장별으로 배출량 합계
###########################################################
#지역별 규모와 평균계싼
region_summary_df = (
    business_df
    .groupby("제공기관명", as_index=False)
    .agg(
        총배출량=("사업장별배출량", "sum"),
        사업장수=("사업장명", "nunique"),
        사업장당평균=("사업장별배출량", "mean"),
        사업장당중앙값=("사업장별배출량", "median")
    )
)

region_summary_df["사업장당총량"] = (
    region_summary_df["총배출량"]
    / region_summary_df["사업장수"]
)
region_summary_df #지역별 규모와 평균을 함께 계산

,제공기관명,총배출량,사업장수,사업장당평균,사업장당중앙값,사업장당총량
0,강원특별자치도 정선군,1.096697e+05,138,794.707942,16.260,794.707942
1,경기도 군포시,1.579872e+07,125,126389.759856,12000.000,126389.759856
2,경기도 부천시,1.546462e+06,154,10041.961779,412.500,10041.961779
3,경기도 수원시,1.748529e+07,193,90597.352332,14201.000,90597.352332
4,경상남도 합천군,1.029182e+05,154,668.299932,0.522,668.299932
5,대구광역시 달서구,7.374182e+07,1185,62229.381993,250.000,62229.381993
6,대구광역시 중구,7.376535e+06,469,15728.219616,202.000,15728.219616
7,부산광역시 금정구,7.675175e+04,332,231.179970,20.000,231.179970
8,전북특별자치도 부안군,6.254900e+04,111,563.504505,200.000,563.504505
9,전북특별자치도 진안군,3.623532e+06,101,35876.554455,150.000,35876.554455


In [ ]:
fig = px.scatter(
    region_summary_df,
    x="사업장수",
    y="총배출량",
    size="사업장당평균",
    color="사업장당중앙값",
    hover_name="제공기관명",
    text="제공기관명",
    title="사업장 수와 총배출량의 관계",
    labels={
        "사업장수": "사업장 수",
        "총배출량": "총 배출량",
        "사업장당평균": "사업장당 평균 배출량",
        "사업장당중앙값": "사업장당 중앙 배출량"
    },
    color_continuous_scale="Reds",
    template="plotly_white"
)

fig.update_traces(textposition="top center")
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")

fig.update_layout(
    width=900,
    height=600,
    title_x=0.5
)

fig.show() # 사업장 수의 영향성, 대형 사업장의 영향성, 소수 극단값의 영향성


In [51]:
top_mean_df = (
    region_summary_df
    .sort_values("사업장당평균", ascending=False)
    .head(10)
    .sort_values("사업장당평균")
)

fig = px.bar(
    top_mean_df,
    x="사업장당평균",
    y="제공기관명",
    orientation="h",
    text_auto=".1f",
    title="사업장당 평균 배출량 Top 10",
    labels={
        "제공기관명": "제공기관",
        "사업장당평균": "사업장당 평균 배출량"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(title_x=0.5)

fig.show()


######################################## 사업장의 중앙값########################################
top_median_df = (
    region_summary_df
    .sort_values("사업장당중앙값", ascending=False)
    .head(10)
)
top_median_df


,제공기관명,총배출량,사업장수,사업장당평균,사업장당중앙값,사업장당총량
3,경기도 수원시,1.748529e+07,193,90597.352332,14201.00,90597.352332
1,경기도 군포시,1.579872e+07,125,126389.759856,12000.00,126389.759856
2,경기도 부천시,1.546462e+06,154,10041.961779,412.50,10041.961779
10,제주특별자치도 서귀포시,2.953944e+05,148,1995.907865,310.00,1995.907865
5,대구광역시 달서구,7.374182e+07,1185,62229.381993,250.00,62229.381993
6,대구광역시 중구,7.376535e+06,469,15728.219616,202.00,15728.219616
8,전북특별자치도 부안군,6.254900e+04,111,563.504505,200.00,563.504505
9,전북특별자치도 진안군,3.623532e+06,101,35876.554455,150.00,35876.554455
7,부산광역시 금정구,7.675175e+04,332,231.179970,20.00,231.179970
0,강원특별자치도 정선군,1.096697e+05,138,794.707942,16.26,794.707942


In [52]:
corr_df = region_summary_df[
    [
        "총배출량",
        "사업장수",
        "사업장당평균",
        "사업장당중앙값"
    ]
].corr()

fig = px.imshow(
    corr_df,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="지역별 배출량 지표 상관관계",
    template="plotly_white"
)

fig.update_layout(title_x=0.5)
fig.show()######################### 상관관계 히트맵 지역별 요약 숫자들간에 관계 파악

In [53]:
daegu_category_df = (
    clean_df[
        clean_df["제공기관명"].str.contains("대구",
        na=False)
    ]
    .groupby("폐기물분류명", as_index=False)
    .agg(
        배출량=("배출량", "sum")
    )
    .sort_values("배출량", ascending=False)
    .head(10)
)

fig = px.bar(
    daegu_category_df,
    x="배출량",
    y="폐기물분류명",
    orientation="h",
    text_auto=".1f",
    title="대구광역시 폐기물 종류별 배출량",
    template="plotly_white"
)

fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
## 사업장이 많으면 어떻게 처리해야 하나?

사업장이 많다는 이유로 대구 데이터를 제거하면 안 됩니
다. 분석 목적에 따라 지표를 나누어야 합니다.

지역 전체 관리 규모를 알고 싶다
→ 총배출량 사용

사업장 하나의 일반적인 수준을 비교하고 싶다
→ 사업장당 평균·중앙값 사용

소수 대형 사업장의 영향이 궁금하다
→ Boxplot·상위 사업장 비중 사용

폐기물 구성 차이를 알고 싶다
→ 폐기물 종류별 누적 막대그래프·Heatmap 사용

따라서 최종 보고서에는 다음 지표를 함께 제시하는 것이
가장 안전합니다.

총배출량
사업장수
사업장당 평균배출량
사업장당 중앙배출량
상위 사업장 집중도

#####################현재 단계에서 우선순위는 다음과 같습니다.######################

1. 사업장 수-총배출량 산점도
2. 사업장당 평균·중앙값 비교
3. 대구 사업장별 Boxplot
4. 대구 폐기물 종류별 막대그래프
5. 마지막에 상관관계 히트맵

핵심은 대구를 “이상치라서 제거할 대상”으로 보기보다, 사
업장 수 효과와 대형 사업장 효과를 분리해서 설명할 대상
으로 보는 것입니다.

In [59]:
# --------------------------------
# 1. 사업장 단위로 먼저 집계
# --------------------------------
business_df = (
    clean_df.dropna(subset=["제공기관명", "사업장명", "배출량"])
    .groupby(
        ["제공기관명", "사업장명"],
        as_index=False
    )
    .agg(
        사업장별배출량=("배출량", "sum")
    )
)

# --------------------------------
# 2. 제공기관별 요약
# --------------------------------
region_summary_df = (
    business_df
    .groupby("제공기관명", as_index=False)
    .agg(
        총배출량=("사업장별배출량", "sum"),
        사업장수=("사업장명", "nunique"),
        사업장당평균=("사업장별배출량", "mean"),
        사업장당중앙값=("사업장별배출량", "median")
    )
)

# 대구 포함 여부
region_summary_df["대구여부"] = (
    region_summary_df["제공기관명"]
    .str.contains("대구", na=False)
)

# --------------------------------
# 3. 대구 사업장 데이터
# --------------------------------
daegu_business_df = business_df[
    business_df["제공기관명"].str.contains("대구",
    na=False)
].copy()

daegu_business_df = daegu_business_df[
    daegu_business_df["사업장별배출량"] > 0
].copy()

# --------------------------------
# 4. 총배출량 상위 제공기관
# --------------------------------
top10_region_df = (
    region_summary_df
    .sort_values("총배출량", ascending=False)
    .head(10)
    .sort_values("사업장당중앙값")
)

# --------------------------------
# 5. 3개 그래프 배치
# --------------------------------
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "사업장 수와 총배출량",
        "대구 사업장별 배출량 분포",
        "평균과 중앙값 비교"
    ],
    horizontal_spacing=0.08
)

# =================================
# 그래프 1
# 사업장 수가 많아서 총량이 높은지 확인
# =================================
normal_region_df = region_summary_df[
    ~region_summary_df["대구여부"]
]

daegu_region_df = region_summary_df[
    region_summary_df["대구여부"]
]

fig.add_trace(
    go.Scatter(
        x=normal_region_df["사업장수"],
        y=normal_region_df["총배출량"],
        mode="markers",
        name="기타 지역",
        text=normal_region_df["제공기관명"],
        hovertemplate=(
            "%{text}<br>"
            "사업장 수: %{x}<br>"
            "총배출량: %{y:,.1f}"
            "<extra></extra>"
        ),
        marker=dict(
            size=9,
            color="#4C78A8",
            opacity=0.7
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=daegu_region_df["사업장수"],
        y=daegu_region_df["총배출량"],
        mode="markers+text",
        name="대구",
        text=daegu_region_df["제공기관명"],
        textposition="top center",
        hovertemplate=(
            "%{text}<br>"
            "사업장 수: %{x}<br>"
            "총배출량: %{y:,.1f}"
            "<extra></extra>"
        ),
        marker=dict(
            size=14,
            color="#E45756",
            line=dict(width=2, color="black")
        )
    ),
    row=1,
    col=1
)

# =================================
# 그래프 2
# 일부 대형 사업장이 전체를 끌어올리는지 확인
# =================================
fig.add_trace(
    go.Box(
        y=daegu_business_df["사업장별배출량"],
        name="대구",
        boxpoints="all",
        jitter=0.35,
        pointpos=0,
        marker=dict(
            color="#E45756",
            size=5
        ),
        line=dict(color="#9E2A2B"),
        hovertext=daegu_business_df["사업장명"],
        hovertemplate=(
            "사업장: %{hovertext}<br>"
            "배출량: %{y:,.1f}"
            "<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# =================================
# 그래프 3
# 평균과 중앙값 비교
# =================================
fig.add_trace(
    go.Bar(
        x=top10_region_df["사업장당평균"],
        y=top10_region_df["제공기관명"],
        orientation="h",
        name="사업장당 평균",
        marker_color="#F58518",
        hovertemplate=(
            "%{y}<br>"
            "평균: %{x:,.1f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=3
)

fig.add_trace(
    go.Bar(
        x=top10_region_df["사업장당중앙값"],
        y=top10_region_df["제공기관명"],
        orientation="h",
        name="사업장당 중앙값",
        marker_color="#54A24B",
        hovertemplate=(
            "%{y}<br>"
            "중앙값: %{x:,.1f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=3
)

# --------------------------------
# 축 설정
# --------------------------------
fig.update_xaxes(
    title_text="사업장 수",
    type="log",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="총배출량",
    type="log",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="사업장별 배출량",
    type="log",
    row=1,
    col=2
)

fig.update_xaxes(
    title_text="배출량",
    type="log",
    row=1,
    col=3
)

# --------------------------------
# 전체 레이아웃
# --------------------------------
fig.update_layout(
    width=1500,
    height=600,
    title="대구광역시 총배출량 원인 분석",
    title_x=0.5,
    template="plotly_white",
    barmode="group",
    legend=dict(
        orientation="h",
        y=-0.15
    )
)

fig.show()